# Lesson 1.1 — LLM Fundamentals: Tokens, Context Windows, and Sampling

**Level: Beginner**

## Learning Objectives

By the end of this lesson, you will be able to:

- Explain tokens, context windows, and next-token generation in Claude
- Describe why LLM output is non-deterministic and how sampling controls it
- Estimate token counts and reason about context-window limits
- Relate temperature and top_p to output variability

**You will produce:** A token-counting + sampling-experiment script using the Messages API (with offline mock).

## Phase 1 Overview

Phase 1 grounds you in how Claude actually works and how to drive it through the Messages API. It maps to **Domain 5 - Model Selection and Optimization (16.8%)** and the **Claude API Mechanics** skill of **Domain 2 - Applications and Integration (33.1%)**. You will move from raw LLM concepts to confident, cost-aware API usage.

By the end of this phase, you will have built:

- A sampling/token experimentation harness
- Model-selection and effort/thinking demos
- A streaming + vision + prompt-caching client
- A batch-vs-realtime decision helper and a token cost calculator

### Prerequisites

- Python 3 and pip
- `pip install anthropic` (SDK that wraps the REST API)
- An optional `ANTHROPIC_API_KEY` env var (all cells run offline without it)
- 6+ months hands-on Claude experience

## Concepts

A **large language model (LLM)** predicts the next **token** given the tokens before it. Text is split into tokens (~4 characters of English on average), and the model generates one token at a time, feeding each new token back in - this is **autoregressive next-token generation**.

- **Context window**: the maximum number of tokens (input + output) the model can attend to in one request. Claude models expose large context windows (commonly 200K tokens; confirm the exact size per model on the docs). Exceed it and you get an error or truncation.
- **Non-determinism**: because the next token is *sampled* from a probability distribution, the same prompt can yield different completions. `temperature` (0-1) flattens or sharpens that distribution; `top_p` does nucleus sampling. `temperature=0` is the closest to deterministic but is **not** guaranteed identical across runs or model versions.
- **Tokens drive cost and latency**: you pay per input and output token, and longer sequences take longer. Token budgeting (Lesson 1.6) starts here.

> **Reference:** [Anthropic - Models & token counting](https://docs.anthropic.com/en/docs/about-claude/models)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — Count tokens before you send

Use the SDK's token counting endpoint to measure prompt size *before* a billed call. This is the foundation of token budgeting.

In [ ]:
prompt = "Explain next-token generation in one sentence."

def count_tokens(text):
    if not HAVE_API:
        # Rough offline heuristic: ~4 chars per token
        return max(1, len(text) // 4)
    resp = client.messages.count_tokens(
        model=MODEL_HAIKU,
        messages=[{"role": "user", "content": text}],
    )
    return resp.input_tokens

print("Estimated input tokens:", count_tokens(prompt))

## Code-Along: Step 2 — Observe non-determinism via sampling

Run the same prompt twice at a higher temperature and compare. With a live key you will often see different wording; the mock path shows the control flow.

In [ ]:
def generate(prompt, temperature=1.0):
    if not HAVE_API:
        return mock_text(prompt)
    msg = client.messages.create(
        model=MODEL_HAIKU,
        max_tokens=60,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

p = "Give a creative two-word name for a coffee shop."
print("Run 1:", generate(p, temperature=1.0))
print("Run 2:", generate(p, temperature=1.0))

## Code-Along: Step 3 — Compare low vs high temperature

Lower temperature concentrates probability on the likeliest tokens (more focused, repeatable).

In [ ]:
print("temp=0.0:", generate(p, temperature=0.0))
print("temp=1.0:", generate(p, temperature=1.0))

## Exercise

Write a function `fits_in_context(text, limit=200_000)` that returns whether a prompt fits a given context window, using `count_tokens`. Then test it on a long string.

In [ ]:
def fits_in_context(text, limit=200_000):
    # TODO: use count_tokens(text) and compare against limit
    # TODO: return a tuple (fits: bool, tokens: int, limit: int)
    pass

# TODO: test with a short string and a very long one (e.g. 'word ' * 100000)

## Key Takeaways

- Claude generates text one token at a time by sampling from a probability distribution.
- The context window bounds input+output tokens; measure size with count_tokens before sending.
- temperature and top_p control variability; temperature=0 is closest to - but not guaranteed - deterministic.
- Tokens drive both cost and latency, so counting them is the first optimization skill.

---

**Next:** [Lesson 1.2 — Model Options: Fast, Extended, and Adaptive Thinking](./lesson-1.2-model-options-thinking.ipynb)